# SeineCrops — Sprint S5 : Service (API)

Prototypage, section par section, de la couche de données de l'API
avant portage vers `src/api/` (FastAPI). L'API expose pour chaque
parcelle sa classe prédite (S3), son statut de divergence (S4), ses
métriques phénologiques (S4) et sa série temporelle d'indices, pour
alimenter la fiche parcelle et le graphique de la carte web.

---

## Structure du notebook (complétée section par section)

| Section | Contenu |
|---------|---------|
| 6.1 | Connexion PostGIS (`asyncpg`) et vérification des tables sources |
| 6.2 | Requête JOIN multi-tables et assemblage de la fiche parcelle |
| 6.3 | Validation du schéma Pydantic `ParcelleDetail` sur données réelles |
| 6.4 | Requête et pivot des indices temporels (`ParcelleProfil`) |
| 6.5 | Diagnostic de densité de parcelles (dimensionnement `bbox`/`limit`) |
| 6.6 | Garde-fou de taille de `bbox` (`HTTP 400`) |
| 6.7 | Requête `bbox` : intersection, simplification, GeoJSON, troncature |
| 6.8 | Fusion garde-fou + requête (version portée vers `src/api/`) |

---

## Références

- `cadrage/methode.md` — §S5 Service (brouillon)
- `04_classification.ipynb` — sprint S3, table `derived.parcelles_classification`
- `05_divergence_pheno.ipynb` — sprint S4, tables `derived.divergence`, `derived.phenologie`

### 6.1 — Connexion PostGIS et vérification des tables sources

Validation de la connexion `asyncpg` (driver retenu pour l'API, cf.
`methode.md` §S5 — évite de bloquer l'event loop FastAPI, contrairement
à `psycopg2` utilisé en synchrone dans les notebooks de pipeline).

Vérification de la présence des 4 tables sources nécessaires à l'API :
`derived.rpg_parcelles_aoi` (S1), `derived.parcelles_classification`
(S3), `derived.divergence` et `derived.phenologie` (S4).

In [ ]:
# ── Imports (communs à toutes les sections) ───────────────────────────
import os
from pathlib import Path

import asyncpg
from dotenv import load_dotenv

# ── Racine du projet ──────────────────────────────────────────────────
def find_project_root(marker: str = ".projectroot") -> Path:
    here = Path().resolve()
    for parent in [here, *here.parents]:
        if (parent / marker).exists() or (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Racine du projet introuvable")

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

# ── Connexion PostGIS (asyncpg — driver retenu pour l'API S5) ──────────
PG_DSN = (
    f"postgresql://{os.getenv('PG_USER', 'postgres')}:{os.getenv('PG_PASSWORD')}"
    f"@{os.getenv('PG_HOST', 'localhost')}:{os.getenv('PG_PORT', 5432)}"
    f"/{os.getenv('PG_DBNAME', 'seinecrops')}"
)

In [ ]:
# --- Vérification de la connexion et des tables sources ---------------

TABLES_ATTENDUES = [
    ("derived", "rpg_parcelles_aoi"),
    ("derived", "parcelles_classification"),
    ("derived", "divergence"),
    ("derived", "phenologie"),
]

conn = await asyncpg.connect(PG_DSN)

for schema, table in TABLES_ATTENDUES:
    existe = await conn.fetchval(
        """
        SELECT EXISTS (
            SELECT 1 FROM information_schema.tables
            WHERE table_schema = $1 AND table_name = $2
        )
        """,
        schema, table,
    )
    n = await conn.fetchval(f"SELECT COUNT(*) FROM {schema}.{table}") if existe else None
    statut = "OK" if existe else "ABSENTE"
    detail = f"{n:,} lignes" if n is not None else ""
    print(f"{schema}.{table:<28} {statut:<8} {detail}")

await conn.close()

### 6.2 — Requête JOIN multi-tables et assemblage de la fiche parcelle

Assemblage de la réponse `GET /parcelles/{id_parcel}` (cf. `methode.md`
§S5) par jointure de quatre tables sur `id_parcel` :
`parcelles_classification` (table de base — 77 932 lignes, même
périmètre que `divergence`/`phenologie`, contrairement à
`rpg_parcelles_aoi` qui inclut les 2 751 parcelles sans pixel 20 m,
cf. notebook 03, section 3.4), `LEFT JOIN` vers `divergence`, `phenologie`
et `rpg_parcelles_aoi` (pour `code_cultu`, le code RPG brut — ex. `BTN` —
distinct de `classe_declaree`, la classe regroupée à 8 valeurs).

**Garde-fou** : les trois tables portent chacune leur propre colonne
`classe_declaree`, issue de la même source RPG mais calculée par des
notebooks différents (nb04, nb05) à des dates différentes — une
vérification de cohérence entre les trois évite qu'un recalcul
partiel (ex. nb05 relancé après un changement de `GROUP_MAP` dans
nb04, sans réexécuter nb04) passe inaperçu.

In [ ]:
# --- Requête JOIN : fiche complète d'une parcelle ---------------------

SQL_FICHE_PARCELLE = """
    SELECT
        c.id_parcel,
        r.code_cultu,
        c.classe_declaree AS classe_declaree_classif,
        c.classe_predite,
        c.proba_max,
        d.classe_declaree AS classe_declaree_div,
        d.dist_classe,
        d.divergent,
        d.zone_raccord_orbital,
        p.classe_declaree AS classe_declaree_pheno,
        p.sos_date,
        p.pos_date,
        p.eos_date,
        p.los_jours,
        p.fiable AS phenologie_fiable
    FROM derived.parcelles_classification c
    LEFT JOIN derived.divergence d USING (id_parcel)
    LEFT JOIN derived.phenologie p USING (id_parcel)
    LEFT JOIN derived.rpg_parcelles_aoi r USING (id_parcel)
    WHERE c.id_parcel = $1
"""

conn = await asyncpg.connect(PG_DSN)

# Parcelle de test : la première de la table de base
id_parcel_test = await conn.fetchval(
    "SELECT id_parcel FROM derived.parcelles_classification LIMIT 1"
)
row = await conn.fetchrow(SQL_FICHE_PARCELLE, id_parcel_test)

await conn.close()

fiche = dict(row)
for k, v in fiche.items():
    print(f"{k:<28} {v}")

In [ ]:
# --- Garde-fou : cohérence de classe_declaree entre les 3 tables ------

SQL_INCOHERENCES = """
    SELECT COUNT(*) FILTER (
        WHERE c.classe_declaree IS DISTINCT FROM d.classe_declaree
           OR c.classe_declaree IS DISTINCT FROM p.classe_declaree
    ) AS n_incoherentes,
    COUNT(*) AS n_total
    FROM derived.parcelles_classification c
    LEFT JOIN derived.divergence d USING (id_parcel)
    LEFT JOIN derived.phenologie p USING (id_parcel)
"""

conn = await asyncpg.connect(PG_DSN)
check = await conn.fetchrow(SQL_INCOHERENCES)
await conn.close()

n_incoherentes, n_total = check["n_incoherentes"], check["n_total"]
print(f"Parcelles avec classe_declaree incohérente entre tables : "
      f"{n_incoherentes:,} / {n_total:,}")
assert n_incoherentes == 0, (
    "Incohérence de classe_declaree entre derived.parcelles_classification, "
    "derived.divergence et derived.phenologie — vérifier si les 3 notebooks "
    "ont bien tourné sur le même run RPG/GROUP_MAP."
)

### 6.3 — Validation du schéma Pydantic `ParcelleDetail`

Instanciation du modèle `ParcelleDetail` (défini dans `methode.md` §S5)
à partir du résultat de la requête JOIN (6.2), avec le renommage
colonne DB → champ API :

| Colonne DB | Champ API |
|---|---|
| `code_cultu` | `code_cultu_declare` |
| `classe_declaree_classif` | `classe_declaree` |
| `proba_max` | `proba_classe` |
| `dist_classe` | `score_divergence` |
| `divergent` | `divergente` |
| `sos_date` / `pos_date` / `eos_date` | `sos` / `pos` / `eos` |
| `fiable` | `phenologie_fiable` |

`classe_declaree_classif` (issue de `derived.parcelles_classification`)
est retenue comme référence, cohérente avec `classe_declaree_div` et
`classe_declaree_pheno` par le garde-fou validé en 6.2.

In [ ]:
# --- Modèle Pydantic ParcelleDetail (cf. methode.md §S5) --------------
from datetime import date
from pydantic import BaseModel


class ParcelleDetail(BaseModel):
    id_parcel: str
    code_cultu_declare: str
    classe_declaree: str
    classe_predite: str
    proba_classe: float
    score_divergence: float
    divergente: bool
    zone_raccord_orbital: bool
    sos: date | None
    pos: date | None
    eos: date | None
    los_jours: int | None
    phenologie_fiable: bool


def ligne_vers_parcelle_detail(row: dict) -> ParcelleDetail:
    return ParcelleDetail(
        id_parcel=row["id_parcel"],
        code_cultu_declare=row["code_cultu"],
        classe_declaree=row["classe_declaree_classif"],
        classe_predite=row["classe_predite"],
        proba_classe=row["proba_max"],
        score_divergence=row["dist_classe"],
        divergente=row["divergent"],
        zone_raccord_orbital=row["zone_raccord_orbital"],
        sos=row["sos_date"],
        pos=row["pos_date"],
        eos=row["eos_date"],
        los_jours=row["los_jours"],
        phenologie_fiable=row["phenologie_fiable"],
    )


fiche_validee = ligne_vers_parcelle_detail(fiche)
print(fiche_validee.model_dump_json(indent=2))

### 6.4 — Requête et pivot des indices temporels (`ParcelleProfil`)

Récupération des 4 indices (NDVI, EVI, NDWI, NDRE) depuis
`derived.s2_parcelles_monthly` (format long — une ligne par
`id_parcel` × `mois` × `variable`, cf. notebook 03) pour la parcelle
test, en amont du pivot vers les listes attendues par `ParcelleProfil`.

Requête brute d'abord, pour vérifier le format de `mois` et le nombre
de lignes réellement présentes (les mois sous le seuil de complétude,
cf. règle 3.5/S2, sont absents plutôt qu'à `NULL` — à confirmer ici
avant d'écrire le pivot).

**Piège identifié (6.4bis)** : les valeurs de `variable` sont stockées
en majuscules (`NDVI`, `EVI`, `NDWI`, `NDRE`, `B02`…) dans
`derived.s2_parcelles_monthly` — l'exemple `ndvi_mean_2024-06` (minuscule)
du Markdown de `04_classification.ipynb` (§4.1) était illustratif, pas
littéral. Corrigé dans `methode.md` §S5 (casse + mapping DB → API).

In [ ]:
# --- Requête brute : indices temporels d'une parcelle -----------------

SQL_PROFIL = """
    SELECT mois, variable, mean
    FROM derived.s2_parcelles_monthly
    WHERE id_parcel = $1 AND variable IN ('NDVI', 'EVI', 'NDWI', 'NDRE')
    ORDER BY mois, variable
"""

conn = await asyncpg.connect(PG_DSN)
rows_profil = await conn.fetch(SQL_PROFIL, id_parcel_test)
await conn.close()

print(f"{len(rows_profil)} lignes (maximum théorique : 16 mois × 4 variables = 64)")
print(f"Mois distincts : {sorted({r['mois'] for r in rows_profil})}")
print()
for r in rows_profil[:8]:
    print(dict(r))

### 6.4bis — Diagnostic : absence inattendue de la parcelle test

0 ligne renvoyée pour `id_parcel_test` alors que cette parcelle est
classifiée (6.2/6.3) — la classification (nb04) est construite depuis
`derived.s2_parcelles_monthly`, donc son absence totale y est suspecte.
Diagnostic avant correction : la parcelle existe-t-elle dans la table
(toutes variables confondues), quelles sont les valeurs réelles de
`variable`, et quel est le type de `id_parcel` ?

In [ ]:
# --- Diagnostic ---------------------------------------------------------

conn = await asyncpg.connect(PG_DSN)

print(f"id_parcel_test = {id_parcel_test!r} (type Python : {type(id_parcel_test)})")

n_toutes_variables = await conn.fetchval(
    "SELECT COUNT(*) FROM derived.s2_parcelles_monthly WHERE id_parcel = $1",
    id_parcel_test,
)
print(f"Lignes pour id_parcel_test (toutes variables) : {n_toutes_variables}")

variables_distinctes = await conn.fetch(
    "SELECT DISTINCT variable FROM derived.s2_parcelles_monthly ORDER BY variable"
)
print(f"Valeurs distinctes de 'variable' en base : "
      f"{[r['variable'] for r in variables_distinctes]}")

types_colonnes = await conn.fetch(
    """
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'derived'
      AND table_name IN ('s2_parcelles_monthly', 'parcelles_classification')
      AND column_name = 'id_parcel'
    """
)
print("Type de id_parcel par table :")
for r in types_colonnes:
    print(f"  {r['table_name']:<28} {r['data_type']}")

await conn.close()

**Pivot vers `ParcelleProfil`** : reconstruction du calendrier de
référence des 16 mois (sept 2023 → déc 2024) et remplissage des mois
absents par `None` (cf. `methode.md` §S5 — les mois sous le seuil de
complétude sont des lignes absentes en base, pas des `NULL`, vérifié
en 6.4bis).

In [ ]:
# --- Pivot vers ParcelleProfil -----------------------------------------
from collections import defaultdict

import pandas as pd


class ParcelleProfil(BaseModel):
    id_parcel: str
    dates: list[date]        # 16 pas mensuels, sept N → déc N+1
    ndvi: list[float | None]
    evi: list[float | None]
    ndwi: list[float | None]
    ndre: list[float | None]


# Calendrier de référence : sept N → déc N+1 (16 mois), aligné sur
# la fenêtre d'observation du projet (cf. methode.md §Zone d'étude)
MOIS_REFERENCE = pd.period_range("2023-09", "2024-12", freq="M").astype(str).tolist()
assert len(MOIS_REFERENCE) == 16, f"Attendu 16 mois, obtenu {len(MOIS_REFERENCE)}"

# variable -> {mois: mean}
valeurs = defaultdict(dict)
for r in rows_profil:
    valeurs[r["variable"]][r["mois"]] = r["mean"]

profil_test = ParcelleProfil(
    id_parcel=id_parcel_test,
    dates=[date.fromisoformat(f"{m}-01") for m in MOIS_REFERENCE],
    ndvi=[valeurs["NDVI"].get(m) for m in MOIS_REFERENCE],
    evi=[valeurs["EVI"].get(m) for m in MOIS_REFERENCE],
    ndwi=[valeurs["NDWI"].get(m) for m in MOIS_REFERENCE],
    ndre=[valeurs["NDRE"].get(m) for m in MOIS_REFERENCE],
)

n_manquants = sum(v is None for v in profil_test.ndvi)
print(f"Mois sans valeur NDVI : {n_manquants} / 16")
print(profil_test.model_dump_json(indent=2))

### 6.5 — Diagnostic de densité de parcelles

Avant d'écrire `GET /parcelles?bbox=`, mesure de la densité moyenne de
parcelles sur l'AOI (`derived.rpg_parcelles_aoi`), pour choisir
`BBOX_MAX_AREA_KM2` et `limit` par défaut sur preuve plutôt qu'a priori
(mêmes principe que la mesure de `tolerance` en 5.x du brouillon
`methode.md` §S5 — les valeurs `50 km²`/`2000` qui y figuraient
n'étaient que des exemples illustratifs, pas des mesures).

In [ ]:
# --- Densité moyenne de parcelles sur l'AOI ----------------------------

SQL_DENSITE = """
    SELECT
        COUNT(*) AS n_parcelles,
        ROUND((SUM(ST_Area(geom)) / 1e6)::numeric, 1) AS surface_totale_km2,
        ROUND((COUNT(*) / (SUM(ST_Area(geom)) / 1e6))::numeric, 1) AS densite_par_km2
    FROM derived.rpg_parcelles_aoi
"""

conn = await asyncpg.connect(PG_DSN)
densite = await conn.fetchrow(SQL_DENSITE)
await conn.close()

for k, v in dict(densite).items():
    print(f"{k:<22} {v}")

### 6.6 — Garde-fou de taille de `bbox`

`BBOX_MAX_AREA_KM2 = 50` (mesuré/décidé en 6.5). Le `bbox` client est
attendu en EPSG:4326 (convention MapLibre/Leaflet) — le calcul de
surface transforme vers EPSG:2154 (Lambert-93, mètres) avant
`ST_Area`, cohérent avec la décision CRS de sortie (`methode.md` §S5).

Test sur deux cas réels tirés de l'emprise de l'AOI : un petit bbox
(~6 km de côté, attendu accepté) et le bbox complet de l'AOI
(3 349 km², attendu refusé).

In [ ]:
# --- Garde-fou de taille de bbox ---------------------------------------

BBOX_MAX_AREA_KM2 = 50

SQL_EXTENT_AOI = """
    SELECT
        ST_XMin(env) AS xmin, ST_YMin(env) AS ymin,
        ST_XMax(env) AS xmax, ST_YMax(env) AS ymax
    FROM (
        SELECT ST_Transform(ST_SetSRID(ST_Extent(geom), 2154), 4326) AS env
        FROM derived.rpg_parcelles_aoi
    ) t
"""

SQL_SURFACE_BBOX = """
    SELECT ROUND(
        (ST_Area(ST_Transform(ST_MakeEnvelope($1, $2, $3, $4, 4326), 2154)) / 1e6)::numeric,
        1
    ) AS surface_km2
"""

conn = await asyncpg.connect(PG_DSN)

extent = await conn.fetchrow(SQL_EXTENT_AOI)
print(f"Emprise AOI (EPSG:4326) : {dict(extent)}")

# Petit bbox de test (~6 km de côté), centré sur l'emprise AOI
centre_lon = (extent["xmin"] + extent["xmax"]) / 2
centre_lat = (extent["ymin"] + extent["ymax"]) / 2
petit_bbox = (centre_lon - 0.03, centre_lat - 0.03, centre_lon + 0.03, centre_lat + 0.03)

grand_bbox = (extent["xmin"], extent["ymin"], extent["xmax"], extent["ymax"])

for nom, bbox in [("petit bbox (~6 km)", petit_bbox), ("bbox complet AOI", grand_bbox)]:
    surface = await conn.fetchval(SQL_SURFACE_BBOX, *bbox)
    verdict = "accepté" if surface <= BBOX_MAX_AREA_KM2 else "refusé (HTTP 400)"
    print(f"{nom:<22} surface = {surface:>8} km²  ->  {verdict}")

await conn.close()

### 6.7 — Requête `bbox` complète

Assemblage de `GET /parcelles?bbox=` : `ST_Intersects` (sélection),
`ST_SimplifyPreserveTopology(geom, 5)` (simplification, `tolerance`
mesurée en amont §S5), puis `ST_Transform` vers EPSG:4326 pour la
sortie GeoJSON — dans cet ordre, la simplification doit s'appliquer
en EPSG:2154 (mètres) *avant* la transformation, pas après.

**Détection de troncature sans `COUNT` systématique** : on demande
`limit + 1` lignes ; si `limit + 1` reviennent, on sait qu'il y en a
plus sans avoir à compter tout l'ensemble — on ne fait un `COUNT(*)`
séparé (pour `total_disponible`) que dans ce cas précis.

In [ ]:
# --- Requête bbox complète ---------------------------------------------
import json as jsonlib

LIMIT_DEFAUT = 2000

SQL_BBOX = """
    SELECT
        c.id_parcel,
        c.classe_predite,
        d.divergent,
        ST_AsGeoJSON(
            ST_Transform(ST_SimplifyPreserveTopology(r.geom, 5), 4326)
        ) AS geometry_json
    FROM derived.parcelles_classification c
    JOIN derived.rpg_parcelles_aoi r USING (id_parcel)
    LEFT JOIN derived.divergence d USING (id_parcel)
    WHERE ST_Intersects(r.geom, ST_Transform(ST_MakeEnvelope($1, $2, $3, $4, 4326), 2154))
    LIMIT $5
"""

SQL_COUNT_BBOX = """
    SELECT COUNT(*)
    FROM derived.parcelles_classification c
    JOIN derived.rpg_parcelles_aoi r USING (id_parcel)
    WHERE ST_Intersects(r.geom, ST_Transform(ST_MakeEnvelope($1, $2, $3, $4, 4326), 2154))
"""


async def fetch_bbox(bbox, limit=LIMIT_DEFAUT):
    conn = await asyncpg.connect(PG_DSN)
    rows = await conn.fetch(SQL_BBOX, *bbox, limit + 1)

    tronque = len(rows) > limit
    rows = rows[:limit]

    total_disponible = len(rows)
    if tronque:
        total_disponible = await conn.fetchval(SQL_COUNT_BBOX, *bbox)

    await conn.close()

    features = [
        {
            "type": "Feature",
            "geometry": jsonlib.loads(r["geometry_json"]),
            "properties": {
                "id_parcel": r["id_parcel"],
                "classe_predite": r["classe_predite"],
                "divergente": r["divergent"],
            },
        }
        for r in rows
    ]

    return {
        "type": "FeatureCollection",
        "features": features,
        "retourne": len(features),
        "total_disponible": total_disponible,
        "tronque": tronque,
    }


resultat = await fetch_bbox(petit_bbox)
print(f"retourne={resultat['retourne']}  total_disponible={resultat['total_disponible']}  "
      f"tronque={resultat['tronque']}")
print("Première feature :")
print(jsonlib.dumps(resultat["features"][0], indent=2)[:600])

**Test de la branche `tronque=True`** (non exercée ci-dessus, puisque
55 < 2000) : même `petit_bbox` (55 parcelles disponibles), avec une
`limit` artificiellement basse.

In [ ]:
# --- Test de troncature (limit artificiellement basse) -----------------

resultat_tronque = await fetch_bbox(petit_bbox, limit=10)
print(f"retourne={resultat_tronque['retourne']}  "
      f"total_disponible={resultat_tronque['total_disponible']}  "
      f"tronque={resultat_tronque['tronque']}")
assert resultat_tronque["retourne"] == 10
assert resultat_tronque["total_disponible"] == 55
assert resultat_tronque["tronque"] is True
print("OK")

### 6.8 — Fusion garde-fou + requête

Une seule fonction `fetch_parcelles_bbox` : vérifie la surface (6.6),
lève une exception dédiée si elle dépasse `BBOX_MAX_AREA_KM2` (portée
vers `HTTPException(400)` côté FastAPI), sinon exécute la requête
(6.7). Test sur `petit_bbox` (accepté) et `grand_bbox` (refus attendu).

In [ ]:
# --- Fonction combinée : garde-fou + requête ----------------------------

class BboxTropLargeError(Exception):
    def __init__(self, surface_km2: float, max_km2: float):
        self.surface_km2 = surface_km2
        self.max_km2 = max_km2
        super().__init__(
            f"bbox trop large (surface {surface_km2} km², maximum {max_km2} km²) "
            f"— réduisez l'emprise géographique demandée"
        )


async def fetch_parcelles_bbox(bbox, limit=LIMIT_DEFAUT):
    conn = await asyncpg.connect(PG_DSN)

    surface = await conn.fetchval(SQL_SURFACE_BBOX, *bbox)
    if surface > BBOX_MAX_AREA_KM2:
        await conn.close()
        raise BboxTropLargeError(surface, BBOX_MAX_AREA_KM2)

    rows = await conn.fetch(SQL_BBOX, *bbox, limit + 1)
    tronque = len(rows) > limit
    rows = rows[:limit]

    total_disponible = len(rows)
    if tronque:
        total_disponible = await conn.fetchval(SQL_COUNT_BBOX, *bbox)

    await conn.close()

    features = [
        {
            "type": "Feature",
            "geometry": jsonlib.loads(r["geometry_json"]),
            "properties": {
                "id_parcel": r["id_parcel"],
                "classe_predite": r["classe_predite"],
                "divergente": r["divergent"],
            },
        }
        for r in rows
    ]

    return {
        "type": "FeatureCollection",
        "features": features,
        "retourne": len(features),
        "total_disponible": total_disponible,
        "tronque": tronque,
    }


# Cas accepté
resultat_ok = await fetch_parcelles_bbox(petit_bbox)
print(f"petit_bbox -> retourne={resultat_ok['retourne']}")

# Cas refusé
try:
    await fetch_parcelles_bbox(grand_bbox)
    print("ERREUR : aurait dû lever BboxTropLargeError")
except BboxTropLargeError as e:
    print(f"grand_bbox -> rejeté comme attendu : {e}")